# Servicing Data Lakes Preparation

**Dataset:** `data/movies.json`, about 3.2 k movies (gross, budget, genre, IMDB and Rotten Tomatoes ratings), one JSON object per line.

### What you will learn
| # | Concept | What you will *see* in this notebook |
|---|---------|--------------------------------------|
| 1 | **Serving layer** | How the data in a lake reaches people and tools that do not write Spark code |
| 2 | **Unmanaged servicing** | Temp views: SQL over files, but only inside *one* Spark application |
| 3 | **Managed servicing** | Tables registered in a shared metastore, created by one application and used by another |
| 4 | **Thrift JDBC/ODBC server** | A long-running Spark application that answers SQL from many clients at once |
| 5 | **Clients** | `beeline` (CLI), Python (`PyHive`), and BI tools such as Tableau or DBeaver |

### Agenda
0. Big picture & setup
1. Unmanaged servicing: temp views and their scope
2. Managed servicing: put the metadata in a metastore
3. Start the Spark Thrift server
4. Connect clients: `beeline`, Python, BI tools
5. The lake stays live: new files, same table
6. Shut down
7. Exercises

> **Requirements:** Spark 4.x installed with `pip install pyspark` (it ships `spark-sql`, `spark-submit`, `beeline` and the Thrift server). Session 25 explains managed vs. external tables and the metastore in depth; this notebook is about **serving** those tables.

## Part 0 — The big picture

A data lake is only useful if people can query it. Analysts use SQL and BI tools, not PySpark. The **serving layer** puts a SQL endpoint in front of the lake:

```
   STORAGE (files)                 SPARK SQL ENGINE                          CLIENTS
 ┌───────────────────┐      ┌────────────────────────────┐        ┌──────────────────────────┐
 │ landing/movies/   │      │  Spark Thrift server       │ JDBC / │ beeline (CLI)            │
 │   *.json          │◄────►│  (HiveServer2 protocol)    │◄──────►│ Python: PyHive           │
 │ spark-warehouse/  │      │  port 10000                │  ODBC  │ Tableau, Power BI,       │
 │   servicing.db/   │      └─────────────┬──────────────┘        │ DBeaver, Excel           │
 └───────────────────┘                    │ "what & where is it?" └──────────────────────────┘
                              ┌───────────▼──────────────┐
                              │ METASTORE ./metastore_db │
                              │ databases, tables, views │
                              └──────────────────────────┘
```

Two ways to serve SQL over the lake:

| | Unmanaged servicing (temp view) | Managed servicing (metastore + server) |
|---|---|---|
| Where the table definition lives | In the memory of one Spark application | In the metastore, shared and persistent |
| Who can query it | Only code in that application | Any client that connects to the server |
| Lifetime | Ends when the application stops | Until someone drops it |
| Typical use | Ad-hoc analysis, ETL steps | Dashboards, reports, analysts, other teams |

### 0.1 Start Spark *without* a Hive metastore

This notebook's own session uses the default **in-memory** catalog. That is on purpose. Locally, the metastore is an embedded **Derby** database, and **only one process can open it at a time**. In Parts 2–5 the metastore belongs to *separate applications*: the `spark-sql` CLI first, then the Thrift server. If this kernel opened `./metastore_db`, it would keep the lock until the kernel dies, even after `spark.stop()`.

> In production the metastore is a shared service (a Hive Metastore on Postgres/MySQL, AWS Glue, Unity Catalog), so many applications use it at the same time.

In [1]:
import pyspark, os, sys, shutil, socket, subprocess, time, pandas as pd
from pathlib import Path
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])

PROJECT_DIR   = Path(os.getcwd())
MOVIES_JSON   = PROJECT_DIR / "data" / "movies.json"
LANDING_DIR   = PROJECT_DIR / "landing"           # raw files as delivered by the source system
WAREHOUSE_DIR = PROJECT_DIR / "spark-warehouse"   # MANAGED tables created by the server
METASTORE_DB  = PROJECT_DIR / "metastore_db"      # embedded Derby metastore (one process at a time!)
LOG_DIR       = PROJECT_DIR / "tmp"

THRIFT_HOST, THRIFT_PORT = "localhost", 10000
JDBC_URL = f"jdbc:hive2://{THRIFT_HOST}:{THRIFT_PORT}"

spark = (SparkSession.builder
         .appName("servicing-data-lakes-preparation")
         .config("spark.ui.showConsoleProgress", "false")      # keep notebook output readable
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark version          :", spark.version)
print("Catalog implementation :", spark.conf.get("spark.sql.catalogImplementation"))   # expect: in-memory
spark

Python /usr/local/bin/python3.12
Java /usr/lib/jvm/java-21-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/11/15 00:46:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version          : 4.3.0
Catalog implementation : in-memory


In [2]:
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

def sql(query, n=100):
    """Run Spark SQL in THIS notebook's session and return the first n rows as pandas."""
    return spark.sql(query).limit(n).toPandas()

# Every separate Spark application below uses the same metastore and warehouse
METASTORE_CONF = [
    "--conf", f"spark.sql.warehouse.dir={WAREHOUSE_DIR.as_posix()}",
    "--conf", f"spark.hadoop.javax.jdo.option.ConnectionURL=jdbc:derby:;databaseName={METASTORE_DB};create=true",
]

def spark_sql_cli(statements):
    """Run SQL in a *separate* Spark application, the spark-sql CLI, exactly like a second program would."""
    cmd = ["spark-sql", "--master", "local[1]", *METASTORE_CONF,
           "--hiveconf", "hive.cli.print.header=true", "-e", statements]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_DIR)
    rows = [l for l in result.stdout.splitlines() if l.strip() and l != "Response code"]
    print("\n".join(rows) or "(no rows)")
    # spark-sql stops at the first failing statement and reports it on stderr as "[ERROR_CLASS] message"
    errors = [l for l in result.stderr.splitlines() if l.startswith("[") or "Exception" in l]
    if errors:
        print("ERROR ->", errors[0][:200])

def port_open(host, port):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex((host, port)) == 0

def tree(path, max_files=3):
    """Print a directory tree (hides _SUCCESS and .crc housekeeping files). max_files=0 -> folders only."""
    path = Path(path)
    if not path.exists():
        print(f"{path}   <-- DOES NOT EXIST")
        return
    for root, dirs, files in os.walk(path):
        dirs.sort()
        depth = len(Path(root).relative_to(path).parts)
        indent = "    " * depth
        print(f"{indent}{Path(root).name}/")
        files = sorted(f for f in files if not f.startswith((".", "_")))
        for f in files[:max_files]:
            print(f"{indent}    {f}  ({(Path(root) / f).stat().st_size:,} bytes)")
        if max_files and len(files) > max_files:
            print(f"{indent}    ... {len(files) - max_files} more file(s)")

## Part 1 — Unmanaged servicing: SQL over files with a temp view

The quickest way to put SQL on a lake: read the files into a DataFrame and register it as a **temporary view**. No metastore, no files copied. The view is only a name for a query plan, held in this application's memory.

In [3]:
movies_df = spark.read.json(str(MOVIES_JSON))          # schema is inferred from the JSON keys
movies_df.createOrReplaceTempView("movies")
movies_df.printSchema()

root
 |-- Creative_Type: string (nullable = true)
 |-- Director: string (nullable = true)
 |-- Distributor: string (nullable = true)
 |-- IMDB_Rating: double (nullable = true)
 |-- IMDB_Votes: long (nullable = true)
 |-- MPAA_Rating: string (nullable = true)
 |-- Major_Genre: string (nullable = true)
 |-- Production_Budget: long (nullable = true)
 |-- Release_Date: string (nullable = true)
 |-- Rotten_Tomatoes_Rating: long (nullable = true)
 |-- Running_Time_min: long (nullable = true)
 |-- Source: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- US_DVD_Sales: long (nullable = true)
 |-- US_Gross: long (nullable = true)
 |-- Worldwide_Gross: long (nullable = true)



In [4]:
sql("SELECT COUNT(*) AS num_movies FROM movies")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,num_movies
0,3201


In [5]:
# Average IMDB rating by genre (movies with no genre are grouped under NULL)
sql("""
SELECT Major_Genre,
       COUNT(*)                   AS num_movies,
       ROUND(AVG(IMDB_Rating), 2) AS avg_rating
FROM movies
GROUP BY Major_Genre
ORDER BY avg_rating DESC
""")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,Major_Genre,num_movies,avg_rating
0,Documentary,43,7.00
1,Western,36,6.84
2,Black Comedy,36,6.82
3,Drama,789,6.77
4,NaN,275,6.50
5,Musical,53,6.45
6,Thriller/Suspense,239,6.36
7,Adventure,274,6.35
8,Concert/Performance,5,6.33
9,Action,420,6.11


### 1.1 Who can see a temp view?

| Object | Visible to | Gone when |
|--------|-----------|-----------|
| `createOrReplaceTempView` | the **SparkSession** that created it | that session ends |
| `createOrReplaceGlobalTempView` | every session of the **same application**, as `global_temp.<name>` | the application stops |
| table in the **metastore** | every application that uses the same metastore | someone drops it |

In [6]:
other_session = spark.newSession()     # same application (same JVM), new session
try:
    other_session.sql("SELECT COUNT(*) FROM movies").show()
except Exception as e:
    print("Temp view in another session ->", str(e).splitlines()[0][:120])

movies_df.createOrReplaceGlobalTempView("movies")
other_session.sql("SELECT COUNT(*) AS seen_from_other_session FROM global_temp.movies").show()

{"ts": "2026-11-15 00:46:13.853", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `movies` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o59.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `movies` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].\nIf you did not qualify the name with a schema, 

Temp view in another session -> [TABLE_OR_VIEW_NOT_FOUND] The table or view `movies` cannot be found. Verify the spelling and correctness of the schema 
+-----------------------+
|seen_from_other_session|
+-----------------------+
|                   3201|
+-----------------------+



A **different application** cannot see either kind of view. We start one with the `spark-sql` command-line tool. It is a separate process and a separate JVM, the same situation as a colleague's program or a BI tool.

In [7]:
spark_sql_cli("SELECT COUNT(*) FROM movies; SELECT COUNT(*) FROM global_temp.movies;")

(no rows)
ERROR -> [TABLE_OR_VIEW_NOT_FOUND] The table or view `movies` cannot be found. Verify the spelling and correctness of the schema and catalog.


**Unmanaged servicing is fine for one program.** It cannot serve a dashboard, a report, or a second team. To share, the table definition must live somewhere *outside* the application. That is the metastore.

## Part 2 — Managed servicing: put the metadata in a metastore

### 2.1 Application 1 creates the metadata

The source system drops its export into `landing/movies/`. One Spark application registers metadata on top of it, then exits:

* `servicing.movies` is an **external** table over the raw JSON files. Spark stores only the schema and the location.
* `servicing.genre_stats` is a **managed** table: a small, pre-aggregated result that Spark writes into the warehouse, ready for a dashboard.

In PySpark, with a Hive-enabled session, the same `movies` table can be created with `spark.catalog.createTable("servicing.movies", path="landing/movies", source="json")`. Here we use SQL, run by the separate `spark-sql` application.

In [8]:
movies_landing = LANDING_DIR / "movies"
movies_landing.mkdir(parents=True, exist_ok=True)
for f in movies_landing.glob("*.json"):            # start clean when the notebook is re-run
    f.unlink()
shutil.copy(MOVIES_JSON, movies_landing / "movies_export_0001.json")   # simulate the source dropping a file
tree(LANDING_DIR)

landing/
    movies/
        movies_export_0001.json  (1,274,347 bytes)


In [9]:
spark_sql_cli(f"""
CREATE DATABASE IF NOT EXISTS servicing
  COMMENT 'Tables served to analysts and BI tools through the Thrift server';

DROP TABLE IF EXISTS servicing.movies;
CREATE TABLE servicing.movies
  USING json
  LOCATION '{movies_landing.as_posix()}'
  COMMENT 'EXTERNAL: raw movie export, read at query time';

DROP TABLE IF EXISTS servicing.genre_stats;
CREATE TABLE servicing.genre_stats
  USING parquet
  COMMENT 'MANAGED: number of movies and average ratings per genre'
AS SELECT COALESCE(Major_Genre, 'Unknown')           AS genre,
          COUNT(*)                                  AS num_movies,
          ROUND(AVG(IMDB_Rating), 2)                AS avg_imdb_rating,
          ROUND(AVG(Rotten_Tomatoes_Rating), 1)     AS avg_rotten_tomatoes_rating
   FROM servicing.movies
   GROUP BY 1;

SHOW TABLES IN servicing;
""")

namespace	tableName	isTemporary
genre_stats
movies


### 2.2 Application 2 uses the metadata

A brand-new application, started later, finds both tables by name. It does not need the file path or the schema, because the metastore knows them.

In [10]:
spark_sql_cli("""
SELECT COUNT(*) AS num_movies FROM servicing.movies;
SELECT * FROM servicing.genre_stats ORDER BY avg_imdb_rating DESC LIMIT 5;
""")

num_movies
3201
genre	num_movies	avg_imdb_rating	avg_rotten_tomatoes_rating
Documentary	43	7.0	81.1
Western	36	6.84	65.0
Black Comedy	36	6.82	58.6
Drama	789	6.77	63.1
Unknown	275	6.5	72.6


In [11]:
# Where the bytes are: external data stays in landing/, managed data is in the warehouse
tree(LANDING_DIR)
tree(WAREHOUSE_DIR / "servicing.db", max_files=1)

landing/
    movies/
        movies_export_0001.json  (1,274,347 bytes)
servicing.db/
    genre_stats/
        part-00000-3903e0ec-49e2-4164-a2ab-e4148b7c3380-c000.snappy.parquet  (1,730 bytes)


## Part 3 — Start the Spark Thrift server

Starting a new application for every query is slow: each `spark-sql` call above took several seconds just to start a JVM. The **Spark Thrift server** is a Spark application that **keeps running** and accepts SQL over the network. It speaks the HiveServer2 protocol, so any JDBC/ODBC client written for Hive can connect to it.

| Step | Full Spark distribution | `pip install pyspark` (this notebook) |
|------|------------------------|---------------------------------------|
| Create metadata | `bin/spark-sql` | `spark-sql` (Part 2) |
| Start the server | `sbin/start-thriftserver.sh` | `spark-submit --class org.apache.spark.sql.hive.thriftserver.HiveThriftServer2 spark-internal` |
| Connect | `bin/beeline -u jdbc:hive2://localhost:10000` | `beeline -u jdbc:hive2://localhost:10000` |

> ⚠️ Run the server from the same folder, with the same metastore settings, as the `spark-sql` step. Otherwise it creates a new, empty metastore and sees no tables.

In [12]:
def start_thrift_server(timeout=120):
    """Launch the Thrift server as a background process and wait until it accepts connections."""
    if port_open(THRIFT_HOST, THRIFT_PORT):
        print(f"Something is already listening on {THRIFT_PORT}; reusing it.")
        return None
    LOG_DIR.mkdir(exist_ok=True)
    log = open(LOG_DIR / "thriftserver.log", "w")
    proc = subprocess.Popen(
        ["spark-submit", "--class", "org.apache.spark.sql.hive.thriftserver.HiveThriftServer2",
         "--name", "Thrift JDBC/ODBC Server", "--master", "local[2]", *METASTORE_CONF,
         "spark-internal", "--hiveconf", f"hive.server2.thrift.port={THRIFT_PORT}"],
        stdout=log, stderr=subprocess.STDOUT, cwd=PROJECT_DIR)
    start = time.time()
    while not port_open(THRIFT_HOST, THRIFT_PORT):
        if proc.poll() is not None:
            raise RuntimeError(f"Thrift server exited; see {LOG_DIR / 'thriftserver.log'}")
        if time.time() - start > timeout:
            raise TimeoutError("Thrift server did not start in time")
        time.sleep(1)
    print(f"Thrift server (pid {proc.pid}) is listening on {JDBC_URL} after {time.time() - start:.0f} s")
    return proc

thrift_proc = start_thrift_server()

Thrift server (pid 685) is listening on jdbc:hive2://localhost:10000 after 5 s


From a terminal you can watch the server's log with `tail -f tmp/thriftserver.log`, and its Spark UI is usually at http://localhost:4041 (4040 is taken by this notebook's session). Each client query appears there as a job, under the **JDBC/ODBC Server** tab.

## Part 4 — Connect clients

### 4.1 `beeline`, the command-line JDBC client

In a terminal, run `beeline -u jdbc:hive2://localhost:10000 -n student` and type `SHOW DATABASES;`, `SHOW TABLES IN servicing;` and so on. The username is free text (there is no authentication on this local server), and the password is empty. Here we run beeline non-interactively with `-e`.

In [13]:
def beeline(statements):
    """Run SQL through the Thrift server with the beeline JDBC client (a separate process)."""
    result = subprocess.run(["beeline", "-u", JDBC_URL, "-n", "student", "--silent=true", "-e", statements],
                            capture_output=True, text=True)
    print(result.stdout.strip())
    errors = [l for l in result.stderr.splitlines() if "Error" in l]
    if errors:
        print("\n".join(errors[:3]))

beeline("SHOW DATABASES; SHOW TABLES IN servicing;")

+------------+
| namespace  |
+------------+
| default    |
| servicing  |
+------------+
+------------+--------------+--------------+
| namespace  |  tableName   | isTemporary  |
+------------+--------------+--------------+
| servicing  | genre_stats  | false        |
| servicing  | movies       | false        |
+------------+--------------+--------------+


In [14]:
beeline("""
SELECT Title, Director, IMDB_Rating
FROM servicing.movies
WHERE IMDB_Rating IS NOT NULL
ORDER BY IMDB_Rating DESC
LIMIT 5;
""")

+---------------------------+-----------------------+--------------+
|           Title           |       Director        | IMDB_Rating  |
+---------------------------+-----------------------+--------------+
| The Godfather             | Francis Ford Coppola  | 9.2          |
| The Shawshank Redemption  | Frank Darabont        | 9.2          |
| Inception                 | Christopher Nolan     | 9.1          |
| The Godfather: Part II    | Francis Ford Coppola  | 9.0          |
| Pulp Fiction              | Quentin Tarantino     | 8.9          |
+---------------------------+-----------------------+--------------+


### 4.2 Python as a client: PyHive

A Python program that only needs **results** does not need Spark at all. It can use a lightweight HiveServer2 driver. This is the Python version of the Java JDBC example:

```java
Connection con = DriverManager.getConnection("jdbc:hive2://localhost:10000", "user", "");
ResultSet rs = con.createStatement().executeQuery("select * from orders limit 10");
```

In [15]:
%pip install -q pyhive thrift thrift-sasl pure-sasl

Note: you may need to restart the kernel to use updated packages.


In [16]:
from pyhive import hive

con = hive.connect(host=THRIFT_HOST, port=THRIFT_PORT, username="student")   # a DB-API 2.0 connection
cur = con.cursor()
cur.execute("SELECT Title, Major_Genre, IMDB_Rating FROM servicing.movies ORDER BY IMDB_Rating DESC LIMIT 10")
for title, genre, rating in cur.fetchall():
    print(f"{rating:>4}  {genre or '-':<20} {title}")

 9.2  -                    The Godfather
 9.2  Drama                The Shawshank Redemption
 9.1  Thriller/Suspense    Inception
 9.0  -                    The Godfather: Part II
 8.9  Drama                12 Angry Men
 8.9  Action               The Dark Knight
 8.9  Drama                Schindler's List
 8.9  -                    One Flew Over the Cuckoo's Nest
 8.9  Drama                Pulp Fiction
 8.9  Adventure            Toy Story 3


In [17]:
def query_server(statement):
    """Run SQL on the Thrift server and return a pandas DataFrame."""
    cur = con.cursor()
    cur.execute(statement)
    columns = [d[0].split(".")[-1] for d in cur.description] if cur.description else []
    return pd.DataFrame(cur.fetchall(), columns=columns)

query_server("SELECT * FROM servicing.genre_stats ORDER BY num_movies DESC")

,genre,num_movies,avg_imdb_rating,avg_rotten_tomatoes_rating
0,Drama,789,6.77,63.1
1,Comedy,675,5.85,47.8
2,Action,420,6.11,48.2
3,Unknown,275,6.50,72.6
4,Adventure,274,6.35,58.6
5,Thriller/Suspense,239,6.36,47.6
6,Horror,219,5.68,41.2
7,Romantic Comedy,137,5.87,43.0
8,Musical,53,6.45,68.0
9,Documentary,43,7.00,81.1


### 4.3 Clients can publish, too

The server is shared, so what one client creates, every other client sees. Here the Python client publishes a **view** for the dashboard team, and `beeline` (a different process) uses it right away.

In [18]:
query_server("""
CREATE OR REPLACE VIEW servicing.director_scorecard
COMMENT 'Directors with at least 5 rated movies'
AS SELECT Director,
          COUNT(*)                   AS num_movies,
          ROUND(AVG(IMDB_Rating), 2) AS avg_imdb_rating,
          SUM(Worldwide_Gross)       AS total_worldwide_gross
   FROM servicing.movies
   WHERE Director IS NOT NULL AND IMDB_Rating IS NOT NULL
   GROUP BY Director
   HAVING COUNT(*) >= 5
""")
beeline("SELECT * FROM servicing.director_scorecard ORDER BY avg_imdb_rating DESC LIMIT 5;")

+--------------------+-------------+------------------+------------------------+
|      Director      | num_movies  | avg_imdb_rating  | total_worldwide_gross  |
+--------------------+-------------+------------------+------------------------+
| Quentin Tarantino  | 6           | 8.25             | 939829449              |
| Christopher Nolan  | 7           | 8.2              | 2409757815             |
| Frank Capra        | 5           | 8.08             | 27100000               |
| Stanley Kubrick    | 5           | 8.04             | 278974927              |
| Peter Jackson      | 6           | 8.02             | 3573395936             |
+--------------------+-------------+------------------+------------------------+


In [19]:
# The view lives in the server's metastore, not in this notebook's in-memory catalog
print("Visible in this notebook's session:", spark.catalog.tableExists("servicing.director_scorecard"))

Visible in this notebook's session: False


### 4.4 BI tools (Tableau, Power BI, DBeaver)

Point the tool at the same server:

| Setting | Value |
|---------|-------|
| Connector | Tableau: *Spark SQL* · DBeaver: *Apache Spark* or *Apache Hive* · Power BI: *Spark* |
| Server / port | `localhost` / `10000` |
| Type / protocol | SparkThriftServer (HiveServer2), binary transport |
| Authentication | Username only (any name), no password |
| JDBC URL | `jdbc:hive2://localhost:10000/servicing` |

> **Running in Docker?** The tool runs on your laptop, so the container must publish the port: add `-p 10000:10000` to your `docker run` command.

## Part 5 — The lake stays live: new files, same table

`servicing.movies` is an external table over a folder, so serving it never copies data. When the source drops a new file, every client sees it after a `REFRESH TABLE`. Spark caches the file list per table, which is why the refresh is needed. The managed `genre_stats` table is a **snapshot**: it changes only when someone rebuilds it.

In [20]:
new_movies = pd.DataFrame([
    {"Title": "Lakehouse Nights", "Major_Genre": "Drama",  "IMDB_Rating": 8.1, "Director": "A. Student"},
    {"Title": "The Last Partition", "Major_Genre": "Thriller/Suspense", "IMDB_Rating": 7.4, "Director": "A. Student"},
])
new_movies.to_json(movies_landing / "movies_export_0002.json", orient="records", lines=True)
tree(movies_landing)

print("Before REFRESH:", query_server("SELECT COUNT(*) AS n FROM servicing.movies")["n"][0])
query_server("REFRESH TABLE servicing.movies")
print("After REFRESH :", query_server("SELECT COUNT(*) AS n FROM servicing.movies")["n"][0])
query_server("SELECT Title, Major_Genre, IMDB_Rating FROM servicing.movies WHERE Director = 'A. Student'")

movies/
    movies_export_0001.json  (1,274,347 bytes)
    movies_export_0002.json  (201 bytes)
Before REFRESH: 3201


After REFRESH : 3203


,Title,Major_Genre,IMDB_Rating
0,Lakehouse Nights,Drama,8.1
1,The Last Partition,Thriller/Suspense,7.4


`Before REFRESH` may already show the new count if no query had cached the file list yet. The `REFRESH` is what makes the result reliable.

## Part 6 — Shut down

Close the client connection and stop the server. Stopping the server releases the Derby lock on `./metastore_db`.

In [21]:
con.close()
if thrift_proc is not None:
    thrift_proc.terminate()
    thrift_proc.wait(timeout=60)
    print("Thrift server stopped")
else:
    print("This notebook did not start the server; stop it where it was started.")

spark.stop()

Thrift server stopped
